# Capítulo 07: Redes Convolucionais Temporais - Soluções dos Exercícios

**Livro:** Análise de Dados e Previsão de Séries Temporais com ML e Sistemas Híbridos Inteligentes

Este notebook contém as soluções completas dos exercícios propostos no Capítulo 07, cobrindo:
- Implementação de TCN com convoluções causais e dilatadas
- Cálculo de campo receptivo
- Comparação TCN vs LSTM
- Arquiteturas híbridas CNN-LSTM

**Nota sobre os dados da HMD:** o fluxo principal deste notebook usa arquivos pré-processados e versionados no repositório do livro. Não dependemos de pacote Python para importar a HMD diretamente. Para quem quiser reproduzir a extração bruta, o notebook do capítulo pode incluir um bloco opcional em **R** com a rotina de importação.
---

In [ ]:
# Configuração inicial e imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import torch
import torch.nn as nn
import torch.nn.functional as F
import time
import warnings
warnings.filterwarnings('ignore')

# Configuração para reprodutibilidade
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')
print('Ambiente configurado com sucesso!')

---

## Exercício 7.1: Implementação de TCN

**Enunciado:** Implemente uma rede convolucional temporal (TCN) em PyTorch para previsão de séries temporais. A arquitetura deve incluir: (a) convoluções causais, (b) convoluções dilatadas com fator de dilatação crescente (1, 2, 4, 8, ...), (c) conexões residuais.

In [ ]:
# =============================================================================
# EXERCÍCIO 7.1: Implementação de TCN
# =============================================================================

# Convolução causal 1D
class CausalConv1d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation=1):
        super(CausalConv1d, self).__init__()
        self.padding = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(
            in_channels, out_channels, kernel_size,
            padding=self.padding, dilation=dilation
        )
    
    def forward(self, x):
        out = self.conv(x)
        # Remove padding do final para manter causalidade
        return out[:, :, :-self.padding]

# Bloco TCN residual
class TCNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation, dropout=0.2):
        super(TCNBlock, self).__init__()
        
        self.conv1 = CausalConv1d(in_channels, out_channels, kernel_size, dilation)
        self.conv2 = CausalConv1d(out_channels, out_channels, kernel_size, dilation)
        self.dropout = nn.Dropout(dropout)
        
        # Conexão residual
        self.downsample = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else None
    
    def forward(self, x):
        residual = x if self.downsample is None else self.downsample(x)
        
        out = F.relu(self.conv1(x))
        out = self.dropout(out)
        out = F.relu(self.conv2(out))
        out = self.dropout(out)
        
        return F.relu(out + residual)

# TCN completa
class TCN(nn.Module):
    def __init__(self, input_size, output_size, num_channels=[64, 64, 64, 64], kernel_size=3, dropout=0.2):
        super(TCN, self).__init__()
        
        layers = []
        num_levels = len(num_channels)
        
        for i in range(num_levels):
            in_channels = input_size if i == 0 else num_channels[i-1]
            out_channels = num_channels[i]
            dilation = 2 ** i
            
            layers.append(TCNBlock(in_channels, out_channels, kernel_size, dilation, dropout))
        
        self.network = nn.Sequential(*layers)
        self.fc = nn.Linear(num_channels[-1], output_size)
    
    def forward(self, x):
        # x: (batch, seq_len, features) -> (batch, features, seq_len)
        x = x.transpose(1, 2)
        out = self.network(x)
        # Pega o último timestep
        out = out[:, :, -1]
        return self.fc(out).squeeze()

print('Classes TCN definidas!')

In [ ]:
# Testar TCN em dados de inflação
np.random.seed(42)
meses = pd.date_range('2000-01-01', '2023-12-01', freq='MS')
n = len(meses)

ipca = np.zeros(n)
ipca[0] = 0.5
for t in range(1, n):
    ar = 0.6 * ipca[t-1] + 0.2 * (ipca[t-2] if t > 1 else ipca[t-1])
    tendencia = 0.3 * np.sin(2 * np.pi * t / 120)
    erro = np.random.normal(0, 0.15)
    ipca[t] = max(0.1, ar + tendencia + erro)

df_ipca = pd.DataFrame({'ipca': ipca}, index=meses)

# Criar sequências
def create_sequences(data, n_lags=24):
    X, y = [], []
    for i in range(n_lags, len(data)):
        X.append(data[i-n_lags:i])
        y.append(data[i])
    return np.array(X), np.array(y)

X, y = create_sequences(df_ipca['ipca'].values, n_lags=24)
X = X.reshape(-1, 24, 1)

# Normalização
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_flat = X.reshape(-1, 24)
X_scaled = scaler_X.fit_transform(X_flat).reshape(-1, 24, 1)
y_scaled = scaler_y.fit_transform(y.reshape(-1, 1)).flatten()

# Split
train_size = int(0.7 * len(X_scaled))
val_size = int(0.15 * len(X_scaled))

X_train = torch.FloatTensor(X_scaled[:train_size]).to(device)
y_train = torch.FloatTensor(y_scaled[:train_size]).to(device)
X_val = torch.FloatTensor(X_scaled[train_size:train_size+val_size]).to(device)
y_val = torch.FloatTensor(y_scaled[train_size:train_size+val_size]).to(device)
X_test = torch.FloatTensor(X_scaled[train_size+val_size:]).to(device)
y_test = y[train_size+val_size:]

print(f'Dados preparados: treino={len(X_train)}, val={len(X_val)}, teste={len(X_test)}')

In [ ]:
# Treinar TCN
model_tcn = TCN(input_size=1, output_size=1, num_channels=[32, 32, 32, 32], kernel_size=3).to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model_tcn.parameters(), lr=0.001)

train_losses = []
val_losses = []
best_val_loss = float('inf')
patience = 15
patience_counter = 0

print('Treinando TCN...')
for epoch in range(100):
    model_tcn.train()
    optimizer.zero_grad()
    outputs = model_tcn(X_train)
    loss = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()
    
    train_losses.append(loss.item())
    
    # Validação
    model_tcn.eval()
    with torch.no_grad():
        val_outputs = model_tcn(X_val)
        val_loss = criterion(val_outputs, y_val).item()
    
    val_losses.append(val_loss)
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = model_tcn.state_dict().copy()
        patience_counter = 0
    else:
        patience_counter += 1
    
    if patience_counter >= patience:
        print(f'Early stopping na época {epoch+1}')
        break

model_tcn.load_state_dict(best_state)

# Avaliar
model_tcn.eval()
with torch.no_grad():
    y_pred_scaled = model_tcn(X_test).cpu().numpy()

y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
mae = mean_absolute_error(y_test, y_pred)

print(f'\nMAE no teste: {mae:.4f}')

---

## Exercício 7.2: Campo Receptivo

**Enunciado:** Investigue o efeito do fator de dilatação no campo receptivo de uma TCN. Derive matematicamente o campo receptivo total.

In [ ]:
# =============================================================================
# EXERCÍCIO 7.2: Campo Receptivo
# =============================================================================

def calculate_receptive_field(kernel_size, num_layers, dilations=None):
    """
    Calcula o campo receptivo de uma TCN.
    
    Fórmula: R = 1 + (kernel_size - 1) * sum(dilations)
    
    Para dilatações em progressão geométrica (1, 2, 4, 8, ...):
    R = 1 + (kernel_size - 1) * (2^num_layers - 1)
    """
    if dilations is None:
        dilations = [2 ** i for i in range(num_layers)]
    
    receptive_field = 1 + (kernel_size - 1) * sum(dilations)
    return receptive_field

# Demonstrar crescimento exponencial
configs = [
    (3, 4),   # kernel=3, 4 camadas
    (3, 6),   # kernel=3, 6 camadas
    (3, 8),   # kernel=3, 8 camadas
    (5, 4),   # kernel=5, 4 camadas
    (5, 6),   # kernel=5, 6 camadas
]

print('=== CAMPO RECEPTIVO DA TCN ===\n')
print(f"{'Kernel':<10} {'Camadas':<10} {'Campo Receptivo':<20} {'Dilatações':<30}")
print("-" * 70)

for kernel, layers in configs:
    rf = calculate_receptive_field(kernel, layers)
    dilations = [2**i for i in range(layers)]
    print(f"{kernel:<10} {layers:<10} {rf:<20} {str(dilations):<30}")

print("\nObservações:")
print('- Com kernel=3 e 8 camadas, o campo receptivo é 511 passos')
print('- O crescimento é exponencial em relação ao número de camadas')
print('- Isso permite capturar dependências de longo prazo com poucas camadas')

In [ ]:
# Visualização do crescimento do campo receptivo
layers_range = range(1, 11)
rf_kernel3 = [calculate_receptive_field(3, l) for l in layers_range]
rf_kernel5 = [calculate_receptive_field(5, l) for l in layers_range]
rf_kernel7 = [calculate_receptive_field(7, l) for l in layers_range]

plt.figure(figsize=(10, 6))
plt.plot(layers_range, rf_kernel3, 'o-', label='Kernel=3', linewidth=2)
plt.plot(layers_range, rf_kernel5, 's-', label='Kernel=5', linewidth=2)
plt.plot(layers_range, rf_kernel7, '^-', label='Kernel=7', linewidth=2)
plt.xlabel('Número de Camadas')
plt.ylabel('Campo Receptivo')
plt.title('Crescimento do Campo Receptivo em TCN')
plt.legend()
plt.grid(True, alpha=0.3)
plt.yscale('log')
plt.tight_layout()
plt.show()

print('\nFórmula geral:')
print('R = 1 + (k - 1) × (2^L - 1)')
print('Onde: k = kernel_size, L = número de camadas')

---

## Exercício 7.3: TCN vs LSTM

**Enunciado:** Teste empiricamente se TCNs superam RNNs em tarefas de sequência onde dependências de longo prazo são importantes.

In [ ]:
# =============================================================================
# EXERCÍCIO 7.3: TCN vs LSTM
# =============================================================================

# Criar série sintética com dependências em múltiplas escalas
np.random.seed(42)
n = 2000

# Componentes em diferentes escalas temporais
curto_prazo = 0.3 * np.random.normal(0, 1, n)  # Ruído de curto prazo
medio_prazo = 0.5 * np.sin(2 * np.pi * np.arange(n) / 50)  # Ciclo de ~50 passos
longo_prazo = 0.8 * np.sin(2 * np.pi * np.arange(n) / 200)  # Ciclo de ~200 passos

# Série combinada
serie = np.zeros(n)
serie[0] = 0
for t in range(1, n):
    autorreg = 0.4 * serie[t-1]
    serie[t] = autorreg + curto_prazo[t] + medio_prazo[t] + longo_prazo[t]

df_serie = pd.DataFrame({'valor': serie})

plt.figure(figsize=(12, 4))
plt.plot(df_serie.index[:500], df_serie['valor'][:500])
plt.title('Série Sintética com Dependências Multi-Escala')
plt.xlabel('Tempo')
plt.ylabel('Valor')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Série criada: {len(df_serie)} observações')
print('Dependências: curto prazo (AR), médio prazo (ciclo 50), longo prazo (ciclo 200)')

In [ ]:
# Preparar dados
def prepare_data(data, n_lags=100):
    X, y = [], []
    for i in range(n_lags, len(data)):
        X.append(data[i-n_lags:i])
        y.append(data[i])
    
    X = np.array(X).reshape(-1, n_lags, 1)
    y = np.array(y)
    
    scaler_X = MinMaxScaler()
    scaler_y = MinMaxScaler()
    
    X_flat = X.reshape(-1, n_lags)
    X_scaled = scaler_X.fit_transform(X_flat).reshape(-1, n_lags, 1)
    y_scaled = scaler_y.fit_transform(y.reshape(-1, 1)).flatten()
    
    train_size = int(0.7 * len(X_scaled))
    val_size = int(0.15 * len(X_scaled))
    
    X_train = torch.FloatTensor(X_scaled[:train_size]).to(device)
    y_train = torch.FloatTensor(y_scaled[:train_size]).to(device)
    X_val = torch.FloatTensor(X_scaled[train_size:train_size+val_size]).to(device)
    y_val = torch.FloatTensor(y_scaled[train_size:train_size+val_size]).to(device)
    X_test = torch.FloatTensor(X_scaled[train_size+val_size:]).to(device)
    y_test = y[train_size+val_size:]
    
    return X_train, y_train, X_val, y_val, X_test, y_test, scaler_y

X_train, y_train, X_val, y_val, X_test, y_test, scaler_y = prepare_data(df_serie['valor'].values)
print('Dados preparados!')

In [ ]:
# Modelo LSTM para comparação
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                           batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :]).squeeze()

# Função de treinamento
def train_model(model, X_train, y_train, X_val, y_val, n_epochs=50):
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    
    best_val_loss = float('inf')
    patience = 10
    patience_counter = 0
    
    start_time = time.time()
    
    for epoch in range(n_epochs):
        model.train()
        optimizer.zero_grad()
        outputs = model(X_train)
        loss = criterion(outputs, y_train)
        loss.backward()
        optimizer.step()
        
        model.eval()
        with torch.no_grad():
            val_outputs = model(X_val)
            val_loss = criterion(val_outputs, y_val).item()
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = model.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1
        
        if patience_counter >= patience:
            break
    
    training_time = time.time() - start_time
    model.load_state_dict(best_state)
    
    return model, training_time, best_val_loss

# Treinar TCN
print('Treinando TCN...')
model_tcn = TCN(input_size=1, output_size=1, num_channels=[32, 32, 32, 32, 32, 32], kernel_size=3).to(device)
model_tcn, time_tcn, val_loss_tcn = train_model(model_tcn, X_train, y_train, X_val, y_val)

# Treinar LSTM
print('Treinando LSTM...')
model_lstm = LSTMModel(input_size=1, hidden_size=64, num_layers=2).to(device)
model_lstm, time_lstm, val_loss_lstm = train_model(model_lstm, X_train, y_train, X_val, y_val)

print('\nTreinamento concluído!')

In [ ]:
# Avaliar no teste
def evaluate_model(model, X_test, y_test, scaler_y):
    model.eval()
    with torch.no_grad():
        y_pred_scaled = model(X_test).cpu().numpy()
    
    y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    return mae, rmse, y_pred

mae_tcn, rmse_tcn, y_pred_tcn = evaluate_model(model_tcn, X_test, y_test, scaler_y)
mae_lstm, rmse_lstm, y_pred_lstm = evaluate_model(model_lstm, X_test, y_test, scaler_y)

# Comparar
print("\n=== COMPARAÇÃO TCN vs LSTM ===")
print(f"\n{'Métrica':<20} {'TCN':<15} {'LSTM':<15} {'Vencedor':<15}")
print("-" * 65)
print(f"{'Tempo de Treino (s)':<20} {time_tcn:<15.3f} {time_lstm:<15.3f} {'TCN' if time_tcn < time_lstm else 'LSTM':<15}")
print(f"{'Val Loss':<20} {val_loss_tcn:<15.6f} {val_loss_lstm:<15.6f} {'TCN' if val_loss_tcn < val_loss_lstm else 'LSTM':<15}")
print(f"{'MAE Teste':<20} {mae_tcn:<15.6f} {mae_lstm:<15.6f} {'TCN' if mae_tcn < mae_lstm else 'LSTM':<15}")
print(f"{'RMSE Teste':<20} {rmse_tcn:<15.6f} {rmse_lstm:<15.6f} {'TCN' if rmse_tcn < rmse_lstm else 'LSTM':<15}")

print("\nConclusão:")
if mae_tcn < mae_lstm:
    print('- TCN supera LSTM em desempenho preditivo')
else:
    print('- LSTM supera TCN em desempenho preditivo')
print(f'- TCN é {time_lstm/time_tcn:.2f}x mais rápido no treinamento')

---

## Exercício 7.4: Arquitetura Híbrida CNN-LSTM

**Enunciado:** Implemente uma arquitetura híbrida CNN-LSTM: camadas convolucionais para extrair features locais, seguidas de LSTM para modelar dependências temporais.

In [ ]:
# =============================================================================
# EXERCÍCIO 7.4: Arquitetura Híbrida CNN-LSTM
# =============================================================================

class CNN_LSTM_Hybrid(nn.Module):
    def __init__(self, input_size, cnn_channels=[32, 64], lstm_hidden=64, lstm_layers=2):
        super(CNN_LSTM_Hybrid, self).__init__()
        
        # CNN layers
        self.conv1 = nn.Conv1d(input_size, cnn_channels[0], kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(cnn_channels[0], cnn_channels[1], kernel_size=3, padding=1)
        self.pool = nn.MaxPool1d(2)
        self.dropout = nn.Dropout(0.2)
        
        # LSTM layers
        self.lstm = nn.LSTM(
            cnn_channels[1], lstm_hidden, lstm_layers,
            batch_first=True, dropout=0.2
        )
        
        # Output
        self.fc = nn.Linear(lstm_hidden, 1)
    
    def forward(self, x):
        # x: (batch, seq_len, features) -> (batch, features, seq_len)
        x = x.transpose(1, 2)
        
        # CNN feature extraction
        x = F.relu(self.conv1(x))
        x = self.dropout(x)
        x = F.relu(self.conv2(x))
        x = self.dropout(x)
        
        # Volta para (batch, seq_len, features) para LSTM
        x = x.transpose(1, 2)
        
        # LSTM
        lstm_out, _ = self.lstm(x)
        
        # Usar último output
        out = self.fc(lstm_out[:, -1, :]).squeeze()
        return out

# Modelo CNN puro para comparação
class CNN_Pure(nn.Module):
    def __init__(self, input_size, channels=[32, 64, 128]):
        super().__init__()
        self.conv1 = nn.Conv1d(input_size, channels[0], 3, padding=1)
        self.conv2 = nn.Conv1d(channels[0], channels[1], 3, padding=1)
        self.conv3 = nn.Conv1d(channels[1], channels[2], 3, padding=1)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(channels[2], 1)
    
    def forward(self, x):
        x = x.transpose(1, 2)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = self.pool(x).squeeze(-1)
        return self.fc(x).squeeze()

print('Modelos híbrido e CNN puro definidos!')

In [ ]:
# Comparar os três modelos
models = {
    'CNN Puro': CNN_Pure(input_size=1).to(device),
    'LSTM': LSTMModel(input_size=1).to(device),
    'CNN-LSTM Híbrido': CNN_LSTM_Hybrid(input_size=1).to(device)
}

results_hybrid = []

for name, model in models.items():
    print(f'\nTreinando {name}...')
    
    model_trained, train_time, val_loss = train_model(model, X_train, y_train, X_val, y_val)
    mae, rmse, _ = evaluate_model(model_trained, X_test, y_test, scaler_y)
    n_params = sum(p.numel() for p in model.parameters())
    
    results_hybrid.append({
        'Modelo': name,
        'Parâmetros': n_params,
        'Tempo (s)': train_time,
        'MAE': mae,
        'RMSE': rmse
    })
    
    print(f'  MAE: {mae:.4f}, RMSE: {rmse:.4f}')

# Resultados
results_hybrid_df = pd.DataFrame(results_hybrid)
print("\n=== COMPARAÇÃO DE ARQUITETURAS ===")
print(results_hybrid_df.to_string(index=False))

best_model = results_hybrid_df.loc[results_hybrid_df['MAE'].idxmin(), 'Modelo']
print(f"\nMelhor modelo: {best_model}")

### Discussão Exercício 7.4

A arquitetura híbrida CNN-LSTM:
- **Vantagem**: Combina extração de features locais (CNN) com modelagem temporal (LSTM)
- **Quando usar**: Quando há padrões locais importantes na série (ex: padrões intradiários)
- **Custo**: Mais parâmetros e tempo de treinamento que modelos puros

---

## Conclusão

Este notebook apresentou soluções para os exercícios do Capítulo 7:

1. **TCN**: Implementação completa com causalidade, dilatação e conexões residuais
2. **Campo Receptivo**: Crescimento exponencial com dilatações em progressão geométrica
3. **TCN vs LSTM**: TCNs são mais rápidas no treino e competitivas em desempenho
4. **Arquiteturas Híbridas**: CNN-LSTM captura o melhor dos dois mundos

Principais takeaways:
- Convoluções causais são essenciais para séries temporais
- Dilatação permite campos receptivos grandes com poucas camadas
- Conexões residuais facilitam o treino de redes profundas